# Python desde cero: la velocidad de las frutas

Vamos a responder una pregunta:

**¿La primera señal recibe una velocidad mayor cuando contiene una naranja que cuando contiene un plátano?**

El archivo contiene **500 respuestas ficticias** con la estructura de una exportación de Prolific.

### Cómo usar el cuaderno

- Ejecuta cada celda con **Mayús + Intro**.
- Lee el comentario que empieza por '#'.
- Cambia solamente colores o números indicados.
- No necesitas memorizar el código: identifica la tabla, la columna y la acción.

## 1. Cargar las librerías

- 'pandas' trabaja con tablas.
- 'numpy' hace operaciones numéricas.
- 'scipy' contiene las pruebas estadísticas.
- 'seaborn' y 'matplotlib' dibujan.

In [ ]:
import math

import pandas as pd
import numpy as np
from scipy import stats
import seaborn as sns
import matplotlib.pyplot as plt

## 2. Calcular el tamaño de la muestra

Queremos detectar un efecto pequeño ('d = 0,25') con potencia 0,80 y un contraste unilateral.

Los valores 1,645 y 0,842 proceden de la distribución normal para alfa 0,05 y potencia 0,80. La fórmula ofrece una aproximación para dos grupos del mismo tamaño.

In [ ]:
d = 0.25

n_grupo = math.ceil(2 * ((1.645 + 0.842) / d) ** 2)
n_total = n_grupo * 2

print("Personas por grupo:", n_grupo)
print("Muestra mínima:", n_total)

**Resultado esperado:** unas 198 personas por grupo y 396 en total. Reclutamos 500 para compensar exclusiones y conservar dos grupos amplios.

## 3. Cargar los datos

El CSV debe estar en la misma carpeta que este cuaderno.

In [ ]:
datos = pd.read_csv("DATOS_FICTICIOS_PROLIFIC_VELOCIDAD_FRUTAS.csv")

datos.head()

In [ ]:
print("Filas:", datos.shape[0])
print("Columnas:", datos.shape[1])

## 4. Preparar una tabla sin identificadores de plataforma

Eliminamos las columnas de Prolific y el texto abierto. Después creamos un identificador docente: 1, 2, 3...

Este procedimiento **minimiza** la tabla de clase, pero no garantiza anonimato irreversible si alguien conserva el CSV original o una tabla de correspondencias.

In [ ]:
datos = datos.drop(columns=[
    "PROLIFIC_PID",
    "STUDY_ID",
    "SESSION_ID",
    "respuesta_abierta"
])

datos["id"] = range(1, len(datos) + 1)

datos.head()

## 5. Mirar los datos con pandas

In [ ]:
# Mostrar cinco filas.
datos.head()

In [ ]:
# Conocer el tamaño de la tabla.
datos.shape

In [ ]:
# Contar valores ausentes.
datos.isna().sum()

In [ ]:
# Calcular la media de la primera respuesta en cada grupo.
datos.groupby("simbolo_primero")["velocidad_primera_kmh"].mean()

## 6. Aplicar la exclusión

La regla se decidió antes de analizar: excluir si cualquiera de las dos respuestas tarda menos de 2 segundos. Exactamente 2,0 segundos permanece.

In [ ]:
validos = datos[
    (datos["tiempo_primera_s"] >= 2.0) &
    (datos["tiempo_segunda_s"] >= 2.0)
].copy()

print("Casos iniciales:", len(datos))
print("Casos excluidos:", len(datos) - len(validos))
print("Casos analizados:", len(validos))

## 7. Resultado principal

Separamos las primeras respuestas en dos grupos y ejecutamos un T-Test de Welch.

- 'equal_var=False': no suponemos varianzas iguales.
- 'alternative="greater"': contrastamos naranja > plátano.

In [ ]:
naranja = validos[
    validos["simbolo_primero"] == "naranja"
]["velocidad_primera_kmh"]

platano = validos[
    validos["simbolo_primero"] == "platano"
]["velocidad_primera_kmh"]

In [ ]:
print("Media naranja:", naranja.mean())
print("Media plátano:", platano.mean())
print("Diferencia:", naranja.mean() - platano.mean())

In [ ]:
resultado_principal = stats.ttest_ind(
    naranja,
    platano,
    equal_var=False,
    alternative="greater"
)

resultado_principal

**Cómo leerlo:** 'statistic' es el estadístico t. 'pvalue' es el p-valor. Si p < 0,05, el resultado apoya la dirección prerregistrada. Conviene mirar también la diferencia de medias.

## 8. Resultado secundario

Comparamos, dentro de cada persona, la velocidad de la naranja y la del plátano. Es exploratorio porque la segunda respuesta puede estar influida por la primera.

In [ ]:
resultado_secundario = stats.ttest_rel(
    validos["velocidad_naranja_kmh"],
    validos["velocidad_platano_kmh"]
)

print("Media naranja:", validos["velocidad_naranja_kmh"].mean())
print("Media plátano:", validos["velocidad_platano_kmh"].mean())
resultado_secundario

## 9. Dibujar el resultado principal

Puedes modificar los dos colores sin cambiar el análisis.

In [ ]:
color_naranja = "#F28C28"
color_platano = "#E5C229"

sns.barplot(
    data=validos,
    x="simbolo_primero",
    y="velocidad_primera_kmh",
    order=["naranja", "platano"],
    hue="simbolo_primero",
    palette={
        "naranja": color_naranja,
        "platano": color_platano
    },
    errorbar=("ci", 95),
    legend=False
)

plt.title("Velocidad asignada en la primera decisión")
plt.xlabel("Primera señal")
plt.ylabel("Velocidad media (km/h)")
plt.xticks([0, 1], ["Naranja", "Plátano"])
plt.ylim(0, 130)
plt.show()

## 10. Las seis instrucciones que debes reconocer

- 'pd.read_csv(...)': cargar una tabla.
- 'datos.head()': mirar las primeras filas.
- 'datos.groupby(...).mean()': calcular medias por grupos.
- 'datos[condición]': filtrar filas.
- 'stats.ttest_ind(...)': comparar dos grupos independientes.
- 'sns.barplot(...)': dibujar una comparación.

Parece indicar que leer Python exige menos memoria que orientación: saber qué objeto entra, qué acción se aplica y qué resultado sale.